In [1]:
import numpy as np
import pandas as pd

In [2]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [3]:
df = pd.read_csv('covid_toy.csv')

In [5]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [6]:
df['cough'].value_counts()

cough
Mild      62
Strong    38
Name: count, dtype: int64

In [7]:
df['city'].value_counts()

city
Kolkata      32
Bangalore    30
Delhi        22
Mumbai       16
Name: count, dtype: int64

In [8]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df.drop(columns= ['has_covid']), df['has_covid'], test_size = 0.2)

In [10]:
X_train

,age,gender,fever,cough,city
53,83,Male,98.0,Mild,Delhi
40,49,Female,102.0,Mild,Delhi
83,17,Female,104.0,Mild,Kolkata
64,42,Male,104.0,Mild,Mumbai
81,65,Male,99.0,Mild,Delhi
...,...,...,...,...,...
52,47,Female,100.0,Strong,Bangalore
33,26,Female,98.0,Mild,Kolkata
21,73,Male,98.0,Mild,Bangalore
84,69,Female,98.0,Strong,Mumbai


1. Without Column Transformer

In [14]:
# adding simple imputer to fever col
si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[['fever']])

# also the test data
X_test_fever = si.transform(X_test[['fever']])

X_train_fever.shape

(80, 1)

In [17]:
# OrdinalEncoding -> Cough
oe = OrdinalEncoder(categories= [['Mild', 'Strong']])
X_train_cough = oe.fit_transform(X_train[['cough']])

# also the test data
X_test_cough = oe.transform(X_test[['cough']])
X_train_cough.shape

(80, 1)

In [23]:
# OneHotEncoding -> gender, city
ohe = OneHotEncoder(drop= 'first')
X_train_gender_city = ohe.fit_transform(X_train[['gender', 'city']]).toarray()

# also the test data
X_test_gender_city = ohe.transform(X_test[['gender', 'city']]).toarray()

X_train_gender_city.shape

(80, 4)

In [27]:
# Extracting age
X_train_age = X_train.drop(columns = ['gender', 'fever', 'cough', 'city']).values

# also the test data
X_test_age = X_test.drop(columns = ['gender', 'fever', 'cough', 'city']).values
X_train_age.shape

(80, 1)

In [30]:
X_train_transformed = np.concatenate((X_train_gender_city, X_train_age, X_train_cough, X_train_fever), axis = 1)
# also the test data
X_test_transformed = np.concatenate((X_test_age, X_test_cough, X_test_fever, X_test_gender_city), axis = 1)

X_train_transformed.shape

(80, 7)

2. Using Column Transformer

In [31]:
from sklearn.compose import ColumnTransformer

In [32]:
transformer = ColumnTransformer(transformers=[
    ('tnf1', SimpleImputer(), ['fever']),
    ('tnf2', OrdinalEncoder(categories=[['Mild', 'Strong']]), ['cough']),
    ('tnf3', OneHotEncoder(drop= 'first'), ['gender', 'city']),
], remainder='passthrough')

In [34]:
transformer.fit_transform(X_train).shape

(80, 7)

In [35]:
transformer.transform(X_test).shape

(20, 7)